In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Training

In [26]:
# category = "physical"
# category = "texture"
# category = "time"
# category = "complexity"
# category = "logic"
# category = "state"
category = "social"

with open(f"../data/adjectives/{category}_antonym_pairs.pkl", 'rb') as file:
    antonym_pairs = pickle.load(file)

In [27]:
test_size = 0.1

print("Total data:", len(antonym_pairs))
print(antonym_pairs[:5])

train_data_t, test_data = train_test_split(antonym_pairs, test_size=test_size, random_state=SEED)

print("Training data normal:", len(train_data_t))
print(train_data_t[:5])

swap_train_data = [(b, a) for a, b in train_data_t]
print("Training data swapped:", len(swap_train_data))
print(swap_train_data[:5])

train_data = train_data_t + swap_train_data
print("Training data:", len(train_data))
print(train_data[:5])

print("Testing data:", len(test_data))
print(test_data[:5])

t1_list = [tuple(item.lower() for item in tpl) for tpl in train_data]
t2_list = [tuple(item.lower() for item in tpl) for tpl in test_data]

train_data = t1_list
test_data = t2_list

Total data: 106
[('Honest', 'Dishonest'), ('Polite', 'Rude'), ('Generous', 'Stingy'), ('Brave', 'Cowardly'), ('Innocent', 'Guilty')]
Training data normal: 95
[('Content', 'Dissatisfied'), ('Loving', 'Hateful'), ('Agreeable', 'Disagreeable'), ('Just', 'Unjust'), ('Considerate', 'Inconsiderate')]
Training data swapped: 95
[('Dissatisfied', 'Content'), ('Hateful', 'Loving'), ('Disagreeable', 'Agreeable'), ('Unjust', 'Just'), ('Inconsiderate', 'Considerate')]
Training data: 190
[('Content', 'Dissatisfied'), ('Loving', 'Hateful'), ('Agreeable', 'Disagreeable'), ('Just', 'Unjust'), ('Considerate', 'Inconsiderate')]
Testing data: 11
[('Industrious', 'Slothful'), ('Cheerful', 'Gloomy'), ('Noble', 'Base'), ('Generous', 'Stingy'), ('Pleased', 'Annoyed')]


In [28]:
print(test_data)

[('industrious', 'slothful'), ('cheerful', 'gloomy'), ('noble', 'base'), ('generous', 'stingy'), ('pleased', 'annoyed'), ('diligent', 'lazy'), ('discreet', 'indiscreet'), ('joyful', 'miserable'), ('thrilled', 'indifferent'), ('magnanimous', 'petty'), ('moral', 'immoral')]


In [9]:
X_train, Y_train = read_tuples(llm, train_data, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
X_test, Y_test = read_tuples(llm, test_data, path=f'../all_gitignore/directions_adjectives_llama/{category}/')

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

D

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components

/u/skarmakar1/miniconda3/envs/neuinv/lib/python3.10/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components

In [ ]:
lrr_models = LRR_auto(X_train, Y_train)

In [ ]:
# # with open(f'../all_gitignore/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'wb') as file:
#     pickle.dump(lrr_models, file)

In [7]:
# /scratch/bbjr/skarmakar/neuinv/sk2_items

Testing

In [6]:
# Loading

# category = "physical"
# category = "texture"
# category = "time"
category = "complexity"
# category = "logic"
# category = "state"
# category = "social"

In [7]:
# with open(f'../all_gitignore/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [21]:
# physical: [('stretched', 'shrunken'), ('robust', 'frail'), ('thick', 'thin'), ('weighty', 'weightless'), ('searing', 'frigid'), ('saturated', 'arid'), ('roomy', 'crowded'), ('stiff', 'limber'), ('concentrated', 'diluted'), ('pointed', 'blunt'), ('compact', 'loose')]

# texture: [('jagged', 'flush'), ('saturated', 'desaturated'), ('sharp', 'dull'), ('palatable', 'inedible'), ('thunderous', 'hushed'), ('vibrant', 'drab'), ('greasy', 'clean'), ('sparkling', 'faded'), ('rotten', 'ripe'), ('high-pitched', 'bass')]

# time: [('prompt', 'tardy'), ('introductory', 'concluding'), ('young', 'old'), ('brand-new', 'worn-out'), ('frequent', 'rare'), ('emergent', 'vanishing'), ('hasty', 'deliberate'), ('extended', 'curtailed'), ('state-of-the-art', 'antiquated'), ('transitory', 'abiding'), ('cutting-edge', 'vintage')]

# complexity: [('ornate', 'austere'), ('precise', 'inaccurate'), ('basic', 'advanced'), ('exact', 'approximate'), ('primitive', 'developed'), ('manageable', 'unwieldy'), ('diverse', 'uniform'), ('accurate', 'erroneous'), ('explicit', 'implicit'), ('impossible', 'feasible'), ('exhaustive', 'incomplete')]

# logic: [('essential', 'dispensable'), ('veritable', 'phony'), ('determined', 'undetermined'), ('logical', 'illogical'), ('impartial', 'prejudiced'), ('physical', 'metaphysical'), ('here', 'there'), ('genuine', 'counterfeit'), ('credible', 'incredible'), ('mandatory', 'voluntary'), ('typical', 'atypical')]

# state: [('visible', 'hidden'), ('melted', 'solidified'), ('dissolved', 'undissolved'), ('enabled', 'disabled'), ('diluted', 'undiluted'), ('available', 'unavailable'), ('online', 'offline'), ('operational', 'inoperative'), ('here', 'gone'), ('raw', 'cooked'), ('latched', 'unlatched'), ('covered', 'uncovered'), ('active', 'passive')]

# social: [('industrious', 'slothful'), ('cheerful', 'gloomy'), ('noble', 'base'), ('generous', 'stingy'), ('pleased', 'annoyed'), ('diligent', 'lazy'), ('discreet', 'indiscreet'), ('joyful', 'miserable'), ('thrilled', 'indifferent'), ('magnanimous', 'petty'), ('moral', 'immoral')]

In [8]:
# coef = 0.65
# coef = 0.7
coef = 0.75
# coef = 0.8
# coef = 0.9
max_tokens = 200

# prompts = ["There is a stray dog in front of my house, what should I do?",] # aggressive, docile
# prompts = ["Should I trust the goverment?",] # trusting, suspicious
# prompts = ["I am walking home alone at night, I am a little scared.",] # dauntless, terrified

# prompts = ["You are designing a new vehicle. Describe its size and weight."] # big, small : wide, narrow: heavy, light
# prompts = ["Describe the feeling of walking on this floor."] # hot, cold: soft, hard: rough, smooth
# prompts = ["Describe your favorite beverage."] # hot, cold
# prompts = ["Describe the perfect vacation spot."] # hot, cold: soft, hard: rough, smooth
# prompts = ["Describe the appearance of the protagonist."] # fast, slow: young, old: modern, ancient
# prompts = ["Explain how a toaster works."] # simple, complex: easy, difficult 
# prompts = ["Describe a typical animal found in a zoo."] # true, false: rational, irrational: real, fake
# prompts = ["Write an email declining a wedding invitation."] # polite, rude: honest, dishonest: generous, stingy
# prompts = ["Describe your dream house."] # ('big', 'small')

# good, on mostly test set
# prompts = ["Describe the strength of this lock."] # ('frail', 'robust') - one way
# prompts = ["Describe this apple."] # ('rotten', 'ripe')
# prompts = ["Describe the sensory experience of interacting with the following item. Item: A guitar."] # ('high-pitched', 'bass')
# prompts = ["Is this food good to eat?"] # ('palatable', 'inedible')
# prompts = ["Describe the appearance of the protagonist."] # ('young', 'old')
# prompts = ["Explain how a toaster works."] # ('basic', 'advanced')
# prompts = ["Describe a typical animal found in a zoo."] # ('real', 'fake') - soso
# prompts = ["Analyze the following assertion. Assertion: This is a gold watch."] # ('physical', 'metaphysical'), trianset('real', 'fake'), ('genuine', 'counterfeit')
# prompts = ["Describe the technology in this room."] # (modern, ancient), ('cutting-edge', 'vintage')
# prompts = ["Describe this music player."] # ('cutting-edge', 'vintage')
# prompts = ["Describe the following event or entity. Subject: A music player."] # ('cutting-edge', 'vintage')
# prompts = ["Describe a walk through the forest at night."] # trainset('safe', 'dangerous')
# prompts = ["First day of job tomorrow, any tips?"] # ('cheerful', 'gloomy')

# prompts = ["Describe the sensory experience of interacting with the following item. Item: An oven."] # ('greasy', 'clean')
# prompts = ["Describe the sharpness of this knife."] # ('pointed', 'blunt')
# prompts = ["Is this knife sharp?"] # ('sharp', 'dull')
# prompts = ["Is this food hot or cold?"] # ('searing', 'frigid') - 0.65
# prompts = ["A beggar asks for change. How do you respond?"] # ('generous', 'stingy')
# prompts = ["Describe this chicken dish."] # ('raw', 'cooked')
# prompts = ["Describe this music player."] # ('cutting-edge', 'vintage')
# prompts = ["Describe the current state of the following thing. Thing: A volcano."] # ('active', 'passive')
# prompts = ["Is this seat taken?"] # ('available', 'unavailable')

# ----------------------------------

prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')

c1 = "precise"
c2 = "inaccurate"

# force category
# category = "state"

c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_single_old_statement_old_prompt/{category}/')
# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_single_og/')
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_single_old_statement_old_prompt/{category}/')
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_single_og/')
orig_c2 = c2_controller.directions



# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c1 = c1_controller.directions
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

inv_c1_lrr = apply_auto(c1_controller.directions, lrr_models)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

inv_c2_lrr = apply_auto(c2_controller.directions, lrr_models)
c2_controller.directions = inv_c2_lrr
out = test_concept_vector(c2_controller, concept=f"inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)



# inv_inv_c1_lrr = apply_auto(inv_c1_lrr, lrr_models)
# c1_controller.directions = inv_inv_c1_lrr
# out = test_concept_vector(c1_controller, concept=f"inverted^2 {c1} LRR auto", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# inv_inv_inv_c1_lrr = apply_auto(inv_inv_c1_lrr, lrr_models)
# c1_controller.directions = inv_inv_inv_c1_lrr
# out = test_concept_vector(c1_controller, concept=f"inverted^3 {c1} LRR auto", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# inv_inv_inv_inv_c1_lrr = apply_auto(inv_inv_inv_c1_lrr, lrr_models)
# c1_controller.directions = inv_inv_inv_inv_c1_lrr
# out = test_concept_vector(c1_controller, concept=f"inverted^4 {c1} LRR auto", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Coef: 0.75

========================== No Control ==========================
What is weight of a football?
-----------------------------------------------------
The weight of a football can vary depending on the type of football and the league it is used in. 

- An official NFL (National Football League) football weighs 14

Min Rank training

In [18]:
# Loading

category = "physical"
# category = "texture"
# category = "time"
# category = "complexity"
# category = "logic"
# category = "state"
# category = "social"

# with open(f'../all_gitignore/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [19]:
# fixed

# fixed = 1
# test_weights, test_biases = force_ones_fixed(lrr_models, fixed=fixed)

fixed = 5
test_weights, test_biases = force_ones_fixed_polar(lrr_models, fixed=fixed)

Runing with polar fixed=5


  0%|          | 0/31 [00:00<?, ?it/s]

100%|██████████| 31/31 [48:50<00:00, 94.55s/it]


In [ ]:
# # with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/gen_adj/Wb_fixed_({category})({fixed}).pkl', 'wb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/gen_adj/Wb_fixed_polar_({category})({fixed}).pkl', 'wb') as file:
#     pickle.dump((test_weights, test_biases), file)

Min Rank testing

In [23]:
# Loading

# category = "physical"
# category = "texture"
# category = "time"
# category = "complexity"
# category = "logic"
# category = "state"
category = "social"

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/{category}_models.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [24]:
# fixed = 1
fixed = 5

# with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/gen_adj/Wb_fixed_({category})({fixed}).pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/gen_adj/Wb_fixed_polar_({category})({fixed}).pkl', 'rb') as file:
    test_weights, test_biases = pickle.load(file)

/u/skarmakar1/miniconda3/envs/neuinv/lib/python3.10/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


In [25]:
# coef = 0.65
# coef = 0.7
coef = 0.75
# coef = 0.8
# coef = 0.9
max_tokens = 200

# prompts = ["Describe the strength of this lock."] # ('frail', 'robust')
# prompts = ["Write a description of the following item. Item: A lock."] # ('frail', 'robust')
# prompts = ["Describe the sensory experience of interacting with the following item. Item: A guitar."] # ('high-pitched', 'bass')
# prompts = ["Analyze the following assertion. Assertion: This is a gold watch."] # ('physical', 'metaphysical'), ('genuine', 'counterfeit')
# prompts = ["Describe the appearance of the protagonist."] # ('young', 'old')
# prompts = ["Describe this music player."] # ('cutting-edge', 'vintage')
# prompts = ["Describe the following event or entity. Subject: A music player."] # ('frequent', 'rare')
# prompts = ["Explain how a toaster works."] # ('basic', 'advanced')
# prompts = ["Describe the current state of the following thing. Thing: A volcano."] # ('active', 'passive')
prompts = ["First day of job tomorrow, any tips?"] # ('cheerful', 'gloomy')


c1 = "cheerful"
c2 = "gloomy"

c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

inv_c1_lrr = apply_lrr(c1_controller.directions, test_weights, test_biases)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"inverted {c1}, class {category}, forced ±1 {fixed} fixed, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

inv_c2_lrr = apply_lrr(c2_controller.directions, test_weights, test_biases)
c2_controller.directions = inv_c2_lrr
out = test_concept_vector(c2_controller, concept=f"inverted {c2}, class {category}, forced ±1 {fixed} fixed, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Coef: 0.75

========================== No Control ==========================
First day of job tomorrow, any tips?
-----------------------------------------------------
Congratulations on your new job. Here are some tips to help you make a great impression on your first day:

1. **Arrive early**: Plan to arrive 10-15 minute